In [1]:
#libraries
import argparse
import os
import random
import subprocess
import mlflow
import mlflow.pytorch
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

/home/vedavachan/aiops-module1-vedavachan/Q2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    lr=0.001,batch_size=32,epochs=5,seed=42)

print("lr:", args.lr)
print("batch_size:", args.batch_size)
print("epochs:", args.epochs)
print("seed:", args.seed)

lr: 0.001
batch_size: 32
epochs: 5
seed: 42


In [ ]:
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

print("Random seeds set =", args.seed)

Random seeds set = 42


In [ ]:
#mlflow configuration
mlflow.set_experiment("Q2-MNIST-MLP")
mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())



Tracking URI: http://127.0.0.1:5000


In [ ]:
#data loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform)

print("training samples:",len(train_dataset))
print("test samples:",len(test_dataset))

training samples: 60000
test samples: 10000


In [ ]:

train_loader = DataLoader(
    train_dataset,
    batch_size=args.batch_size,
    shuffle=True)

val_loader = DataLoader(
    test_dataset,
    batch_size=args.batch_size,
    shuffle=False)

print("Batch size:",args.batch_size)

Batch size: 32


In [ ]:
#model MLP
class MLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),

            nn.Linear(128,64),
            nn.ReLU(),

            nn.Linear(64,10))

    def forward(self, x):
        return self.network(x)


model = MLP()
print(model)

MLP(
  (network): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [ ]:
# training function
def train_one_epoch(
    model,loader,
    criterion,optimizer):

    model.train()
    total_loss = 0.0

    for images, labels in loader:

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()

        total_loss += (loss.item()*images.size(0))

    return (total_loss/len(loader.dataset))

In [ ]:
def evaluate(model,loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images,labels in loader:
            outputs = model(images)
            predictions = torch.argmax(outputs,dim=1)

            correct+=(predictions==labels).sum().item()
            total+=labels.size(0)

    return correct/total

In [ ]:
def get_git_commit_hash():

    try:

        commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.DEVNULL
            ).decode().strip()

        return commit

    except Exception:return "unknown"


git_commit = get_git_commit_hash()


In [ ]:
#experiment function
def run_experiment(
    learning_rate,
    batch_size,
    epochs=5,seed=42):

    #reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    #dataLoaders
    experiment_train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    experiment_val_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False)

    #model
    model = MLP()

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate)

    run_name = (f"MLP-lr{learning_rate}-batch{batch_size}")

    with mlflow.start_run(run_name=run_name):

        #Parameters
        mlflow.log_param(
            "learning_rate",
            learning_rate)

        mlflow.log_param(
            "batch_size",
            batch_size)

        mlflow.log_param(
            "epochs",
            epochs)

        mlflow.log_param(
            "seed",
            seed)

        mlflow.log_param(
            "dataset",
            "MNIST")

        mlflow.log_param(
            "model",
            "MLP")

        # Git commit
        mlflow.set_tag(
            "git_commit",
            git_commit)

        #Training
        for epoch in range(epochs):

            train_loss = train_one_epoch(model,experiment_train_loader,criterion,optimizer)

            val_accuracy = evaluate(model,experiment_val_loader)

            # Metrics
            mlflow.log_metric(
                "train_loss",
                train_loss,
                step=epoch)

            mlflow.log_metric(
                "val_accuracy",
                val_accuracy,
                step=epoch)

            print(
                f"Epoch {epoch + 1}/{epochs} "
                f"| Train Loss: {train_loss:.4f} "
                f"| Val Accuracy: {val_accuracy:.4f}")

        # Save model artifact
        model_path = (
            f"mnist_mlp_lr{learning_rate}_batch{batch_size}.pth")

        torch.save(
            model.state_dict(),
            model_path)

        mlflow.log_artifact(
            model_path)

        print(f"\nCompleted: {run_name}")

        return {
            "run_name": run_name,
            "train_loss": train_loss,
            "val_accuracy": val_accuracy}

In [ ]:
experiments = [
    {"learning_rate": 0.001,
        "batch_size": 32},

    {"learning_rate": 0.001,
        "batch_size": 64},

    {"learning_rate": 0.005,
        "batch_size": 32},
    {"learning_rate": 0.005,
        "batch_size": 64},

    {"learning_rate": 0.01,
        "batch_size": 32},

    {"learning_rate": 0.01,
        "batch_size": 64}]

print("number of experiments:",len(experiments))

for i, experiment in enumerate(experiments,1):
    print(
        f"Experiment {i}: "
        f"lr={experiment['learning_rate']}, "
        f"batch_size={experiment['batch_size']}")

number of experiments: 6
Experiment 1: lr=0.001, batch_size=32
Experiment 2: lr=0.001, batch_size=64
Experiment 3: lr=0.005, batch_size=32
Experiment 4: lr=0.005, batch_size=64
Experiment 5: lr=0.01, batch_size=32
Experiment 6: lr=0.01, batch_size=64


In [ ]:
results = []

for i, experiment in enumerate(experiments, 1):

    print(f"Running Experiment {i}/6")
    print(f"Learning rate: {experiment['learning_rate']}")
    print(f"Batch size: {experiment['batch_size']}")

    result = run_experiment(
        learning_rate=experiment["learning_rate"],
        batch_size=experiment["batch_size"],
        epochs=5,
        seed=42)

    result["learning_rate"] = experiment["learning_rate"]
    result["batch_size"] = experiment["batch_size"]

    results.append(result)

print("\nAll six experiments done")

Running Experiment 1/6
Learning rate: 0.001
Batch size: 32


Epoch 1/5 | Train Loss: 0.2351 | Val Accuracy: 0.9540
Epoch 2/5 | Train Loss: 0.1014 | Val Accuracy: 0.9697
Epoch 3/5 | Train Loss: 0.0752 | Val Accuracy: 0.9728
Epoch 4/5 | Train Loss: 0.0583 | Val Accuracy: 0.9730
Epoch 5/5 | Train Loss: 0.0506 | Val Accuracy: 0.9784

Completed: MLP-lr0.001-batch32
🏃 View run MLP-lr0.001-batch32 at: http://127.0.0.1:5000/#/experiments/1/runs/3631140971954268b197a7c03275dabb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Running Experiment 2/6
Learning rate: 0.001
Batch size: 64
Epoch 1/5 | Train Loss: 0.2646 | Val Accuracy: 0.9539
Epoch 2/5 | Train Loss: 0.1106 | Val Accuracy: 0.9687
Epoch 3/5 | Train Loss: 0.0799 | Val Accuracy: 0.9713
Epoch 4/5 | Train Loss: 0.0607 | Val Accuracy: 0.9708
Epoch 5/5 | Train Loss: 0.0482 | Val Accuracy: 0.9789

Completed: MLP-lr0.001-batch64
🏃 View run MLP-lr0.001-batch64 at: http://127.0.0.1:5000/#/experiments/1/runs/d1f3bbb96edd4f2ab6f73bdda43095cb
🧪 View experiment at: http://127.0.0.1:5000/#/experimen

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df = results_df[
    ["run_name",
        "learning_rate",
        "batch_size",
        "train_loss",
        "val_accuracy"]]

results_df

,run_name,learning_rate,batch_size,train_loss,val_accuracy
0,MLP-lr0.001-batch32,0.001,32,0.050606,0.9784
1,MLP-lr0.001-batch64,0.001,64,0.048249,0.9789
2,MLP-lr0.005-batch32,0.005,32,0.119688,0.9648
3,MLP-lr0.005-batch64,0.005,64,0.090015,0.9642
4,MLP-lr0.01-batch32,0.010,32,0.193372,0.9514
5,MLP-lr0.01-batch64,0.010,64,0.147469,0.9606


In [ ]:
learning_rate_summary = (
    results_df
    .groupby("learning_rate")["val_accuracy"]
    .mean()
    .reset_index())

learning_rate_summary

,learning_rate,val_accuracy
0,0.001,0.97865
1,0.005,0.96450
2,0.010,0.95600


In [ ]:
batch_size_summary = (
    results_df
    .groupby("batch_size")["val_accuracy"]
    .mean()
    .reset_index())

batch_size_summary

,batch_size,val_accuracy
0,32,0.964867
1,64,0.967900
